# LIBERO eval — **최종 표** (상태판단 + SR + 떨림[aloha 방식])

실행하면 **① 학습/eval 됐는지 먼저 판단** → ② 500ep 완료만으로 SR·떨림 표 → ③ zip.

- **떨림은 팀원(은지) aloha 스크립트와 동일 계산식**(`smooth_metrics_paper`): 경계/내부 jerk RMS,
  B/I ratio, SPARC(speed-profile, fs=30), ldj_cost, sign-flip rate. → aloha 와 바로 비교 가능.
- 표기: `bimamba_s7` → **ours**, 이름의 `s7` → **mosaic**.
- 읽기 전용. ①의 결과를 나(클로드)한테 주면 **학습할 것 / (재)eval 할 것** 을 정리해줌.


In [ ]:
import sys, json, csv, time
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np
import smooth_metrics_paper as smp        # 팀원(aloha) 과 동일한 스무스니스 계산식
importlib.reload(smp)

TASK   = 'libero_10'
SEEDS  = [0, 1, 2, 3]
TARGET_EP = cf.EVAL_N_EP                 # 500 — 이 값 이상만 최종 채택
FS     = cf.fps_of(TASK)                 # libero=30 (SPARC 주파수축에만 영향)
STRIDE = 100                             # 청크 경계(하드스위치 기준). aloha 와 동일

# ── 최종 표 모델 (folder_tag, 표기 라벨, 역할) ─  s7→mosaic, bimamba_s7→ours ─
#   ※ 팀과 최종 합의 필요: carry-only / overlap-only 폴더명. 우선 aloha ablation 정렬안.
DESIRED = [
    ('act',        'ACT',                     'baseline'),
    ('acm',        'ACM (Mamba-1, no carry)', 'baseline'),
    ('acm2',       'ACM2 (Mamba-2, no carry)','baseline'),
    ('carry',      'carry only',              'ablation'),   # 폴더명 미확정(미학습 예상)
    ('bimamba',    'BiMamba (carry+BiMamba)', 'ablation'),
    ('overlap',    'mosaic only',             'ablation'),   # 폴더명 미확정(미학습 예상)
    ('bimamba_s7', 'ours',                    'full'),
]
def label(tag):
    for t, lb, _ in DESIRED:
        if t == tag:
            return lb
    return 'ours' if tag == 'bimamba_s7' else tag.replace('s7', 'mosaic')

for t, _, _ in DESIRED:
    cf.v23.MODEL_DIR_NAMES.setdefault(t, t)   # 비표준 태그도 폴더명=태그로 등록

TRAIN_ROOT = cf.OUTPUT_BASE / 'train' / TASK
EVAL_ROOT  = cf.OUTPUT_BASE / 'eval_clean' / TASK
OUT = cf.OUTPUT_BASE / 'share' / f'{TASK}_final'
OUT.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M')
print('train:', TRAIN_ROOT, '| eval:', EVAL_ROOT)
print(f'fs={FS} (SPARC) · 경계 stride={STRIDE} · 최종 채택 {TARGET_EP}ep · 스냅샷 {STAMP}')

## 1) 상태 판단 — 학습됐나 / 500ep eval 됐나 / action 있나
여기서 **학습 필요 / (재)eval 필요 / 준비됨** 으로 자동 분류. carry-only·overlap-only·acm2 는 대개 미학습으로 뜬다.


In [ ]:
# ── 상태 판단: 각 모델×seed 가 학습됐나 / 500ep eval 됐나 / action(.pt) 있나 ──
def eval_rec(tag, seed):
    d = EVAL_ROOT / tag / f'seed{seed}'          # rep0/ 등 하위 포함
    if not d.is_dir():
        return None
    best = None
    for info in d.rglob('eval_info.json'):
        try:
            ov = json.loads(info.read_text()).get('overall', {})
        except Exception:
            continue
        n_ep = ov.get('n_ep', ov.get('n_episodes')) or 0
        has_act = (info.parent / 'actions').is_dir()
        if best is None or n_ep > best['n_ep']:
            best = {'sr': ov.get('pc_success'), 'n_ep': n_ep, 'has_actions': has_act,
                    'path': info.parent}
    return best

def trained(tag, seed):
    return cf.v23.last_ckpt_step(TRAIN_ROOT / tag / f'seed{seed}')

print(f"{'model (라벨)':<26}{'seed':>5}{'학습':>10}{'eval':>10}{'action':>8}")
print('-' * 60)
todo_train, todo_eval, ready = [], [], []
status = {}
for tag, lb, role in DESIRED:
    n_tr = n_ev = n_act = 0
    for s in SEEDS:
        st = trained(tag, s); ev = eval_rec(tag, s)
        ok_tr = st is not None and st >= cf.CKPT_STEP
        ok_ev = ev is not None and (ev['n_ep'] or 0) >= TARGET_EP
        ok_act = bool(ev and ev['has_actions'])
        n_tr += ok_tr; n_ev += ok_ev; n_act += ok_act
        tr_s = f'{st:,}' if st else '-'
        ev_s = f"{ev['n_ep']}ep" if ev else '-'
        print(f"{tag+' ('+lb+')':<26}{s:>5}{tr_s:>10}{ev_s:>10}{'O' if ok_act else '-':>8}")
    status[tag] = {'tr': n_tr, 'ev': n_ev, 'act': n_act}
    if n_tr == 0:
        todo_train.append(tag)
    elif n_ev < len(SEEDS) or n_act < n_ev:
        todo_eval.append(tag)
    else:
        ready.append(tag)

print('\n' + '=' * 60)
print('■ 학습 필요 (체크포인트 없음):        ', [f'{t}({label(t)})' for t in todo_train] or '없음')
print('■ eval/재eval 필요 (학습됐으나 500ep·action 부족):', [f'{t}({label(t)})' for t in todo_eval] or '없음')
print('■ 준비됨 (500ep+action, 표에 반영):   ', [f'{t}({label(t)})' for t in ready] or '없음')

# 폴더에 있는데 DESIRED 에 없는 모델(누락 방지)
on_disk = {p.name for p in TRAIN_ROOT.glob('*') if p.is_dir()} | {p.name for p in EVAL_ROOT.glob('*') if p.is_dir()}
extra = sorted(on_disk - {t for t, _, _ in DESIRED})
if extra:
    print('\n(참고) DESIRED 밖 폴더도 있음:', extra)

## 2) SR 표 (500ep 완료만, mean±std, pooled 95% CI)


In [ ]:
# ── SR 표: 500ep 완료만 · model×seed · mean±std · pooled 95%CI (라벨 적용) ──
hdr = f"{'model':<26}" + ''.join(f'{("s"+str(s)):>7}' for s in SEEDS) + f"{'mean':>8}{'±std':>7}{'pooled95%CI':>16}"
print(hdr); print('-' * len(hdr))
sr_rows = []
for tag, lb, role in DESIRED:
    per, pk, pn = {}, 0, 0
    for s in SEEDS:
        ev = eval_rec(tag, s)
        if ev and (ev['n_ep'] or 0) >= TARGET_EP and ev['sr'] is not None:
            per[s] = ev['sr']; n = int(ev['n_ep']); pk += int(round(ev['sr'] / 100 * n)); pn += n
    vals = list(per.values())
    mean = float(np.mean(vals)) if vals else None
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else (0.0 if vals else None)
    ci = ''
    if pn:
        lo, hi = cf.v23.wilson_ci(pk, pn); ci = f'[{lo*100:.1f},{hi*100:.1f}]'
    cells = ''.join((f'{per[s]:>7.1f}' if s in per else f'{chr(183):>7}') for s in SEEDS)
    mean_s = f'{mean:>8.1f}' if mean is not None else f'{chr(45):>8}'
    std_s = f'{std:>7.1f}' if std is not None else f'{chr(45):>7}'
    print(f'{lb:<26}{cells}{mean_s}{std_s}{ci:>16}')
    sr_rows.append({'model': tag, 'label': lb, 'role': role,
                    **{f'seed{s}': (round(per[s], 1) if s in per else None) for s in SEEDS},
                    'mean': (round(mean, 2) if mean is not None else None),
                    'std': (round(std, 2) if std is not None else None),
                    'n_seed': len(vals),
                    'pooled_sr': (round(pk / pn * 100, 2) if pn else None), 'pooled_ci': ci})
cols = ['model', 'label', 'role'] + [f'seed{s}' for s in SEEDS] + ['mean', 'std', 'n_seed', 'pooled_sr', 'pooled_ci']
with open(OUT / f'sr_table_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(sr_rows)
print('\nsaved:', OUT / f'sr_table_{STAMP}.csv')

## 3) 떨림 표 — aloha 와 동일 (경계/내부 jerk · SPARC · ldj_cost · sign-flip)


In [ ]:
# ── 떨림 표 (팀원=aloha 방식): 경계/내부 jerk RMS · B/I ratio · SPARC · ldj_cost · signflip ──
#   500ep 완료 + action(.pt) 있는 seed 의 궤적을 pool. fs=30, 경계 stride=100.
smooth = {}
for tag, lb, role in DESIRED:
    trajs = []
    for s in SEEDS:
        ev = eval_rec(tag, s)
        if ev and (ev['n_ep'] or 0) >= TARGET_EP and ev['has_actions']:
            trajs += (cf.v23._load_action_trajs(ev['path'] / 'actions') or [])
    if trajs:
        smooth[tag] = smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS)

if not smooth:
    print('action(.pt) 있는 완료 eval 이 없음 — SR 표만. (eval 이 RECORD_DIR 로 궤적 저장했는지 확인)')
else:
    print(f"{'model':<24}{'jerk_RMS':>9}{'bnd_jerk':>9}{'int_jerk':>9}{'B/I':>7}{'SPARC':>9}{'ldj_cost':>9}{'signflip':>9}{'n':>6}")
    print('  방향:            ↓        ↓        ↓     →1     →0        ↓        ↓')
    print('-' * 91)
    srows = []
    for tag, lb, role in DESIRED:
        a = smooth.get(tag)
        if not a:
            continue
        print(f"{lb:<24}{a['jerk_rms_mean']:>9.4f}{a['boundary_jerk_rms_mean']:>9.4f}"
              f"{a['interior_jerk_rms_mean']:>9.4f}{a['boundary_interior_ratio_mean']:>7.2f}"
              f"{a['sparc_mean']:>9.2f}{a['ldj_cost_mean']:>9.2f}{a['sign_flip_rate_mean']:>9.4f}{a['n_traj']:>6}")
        srows.append({'model': tag, 'label': lb,
                      'jerk_rms': round(a['jerk_rms_mean'], 5),
                      'boundary_jerk_rms': round(a['boundary_jerk_rms_mean'], 5),
                      'interior_jerk_rms': round(a['interior_jerk_rms_mean'], 5),
                      'boundary_interior_ratio': round(a['boundary_interior_ratio_mean'], 4),
                      'sparc': round(a['sparc_mean'], 3), 'ldj_cost': round(a['ldj_cost_mean'], 3),
                      'sign_flip_rate': round(a['sign_flip_rate_mean'], 5), 'n_traj': a['n_traj']})
    with open(OUT / f'smoothness_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'label', 'jerk_rms', 'boundary_jerk_rms',
            'interior_jerk_rms', 'boundary_interior_ratio', 'sparc', 'ldj_cost', 'sign_flip_rate', 'n_traj'])
        w.writeheader(); w.writerows(srows)
    print('\nsaved:', OUT / f'smoothness_{STAMP}.csv')

## 4) 최종 표 PNG + zip


In [ ]:
# ── 최종 표 PNG(SR+떨림 한 장) + 요약 MD + zip ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
col = ['SR mean±std', 'n', 'bnd_jerk', 'int_jerk', 'B/I', 'SPARC', 'ldj_cost', 'signflip']
rtxt, rlab = [], []
for r in sr_rows:
    sr = '-' if r['mean'] is None else f"{r['mean']:.1f}±{r['std']:.1f}"
    a = smooth.get(r['model'])
    if a:
        rtxt.append([sr, f"{r['n_seed']}/{len(SEEDS)}", f"{a['boundary_jerk_rms_mean']:.3f}",
                     f"{a['interior_jerk_rms_mean']:.3f}", f"{a['boundary_interior_ratio_mean']:.2f}",
                     f"{a['sparc_mean']:.1f}", f"{a['ldj_cost_mean']:.1f}", f"{a['sign_flip_rate_mean']:.3f}"])
    else:
        rtxt.append([sr, f"{r['n_seed']}/{len(SEEDS)}", '-', '-', '-', '-', '-', '-'])
    rlab.append(r['label'])
fig, ax = plt.subplots(figsize=(2 + 1.3 * len(col), 0.7 + 0.5 * len(rlab)))
ax.axis('off')
t = ax.table(cellText=rtxt, rowLabels=rlab, colLabels=col, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.6)
ax.set_title(f'LIBERO-10 final ({TARGET_EP}ep, aloha-matched smoothness)  {STAMP}', fontsize=11, pad=12)
png = OUT / f'final_table_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight'); plt.close(fig)
print('saved:', png)

lines = [f'# LIBERO-10 최종 ({TARGET_EP}ep, {STAMP})', '',
         f'- 떨림 = aloha 와 동일 스크립트(smooth_metrics_paper), fs={FS}, 경계 stride={STRIDE}',
         f'- 학습 필요: {todo_train}', f'- eval/재eval 필요: {todo_eval}', f'- 준비됨: {ready}']
(OUT / f'README_{STAMP}.md').write_text('\n'.join(lines), encoding='utf-8')
import shutil
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_final_{STAMP}'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path)
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')